## Описание
Реализация на невронна мрежа, която решава булеви функции (AND, OR, XOR) с възможност за добавяне на скрити слоеве, контрол на броя неврони в тях, избор на активационна функция (\(\sigma\) – сигмоидална или \(tanh\)), обучение чрез обратно разпространение на грешката (error backpropagation) и извеждане на резултатите.

Може да се тества и с ALL, където накуп се показват резултатите за AND, OR, XOR.

### Вход:
1. Булева функция: **AND**, **OR**, **XOR** или **ALL** (за да се изкарат резултати за трите).
2. Активационна функция: **0** за сигмоидна, **1** за хиперболичен тангенс.
3. Брой скрити слоеве.
4. Брой неврони във всеки един от скритите слоеве.

### Изход:
За всяка комбинация от входни стойности се извежда изходът на съответната булева функция.

Примерен вход:
```
ALL
0
1
4
```

Примерен изход:
```
AND:
(0,0) -> 0.0199
(0,1) -> 0.0599
(1,0) -> 0.0379
(1,1) -> 0.9589

OR:
(0,0) -> 0.0122
(0,1) -> 0.9899
(1,0) -> 0.9382
(1,1) -> 0.9999

XOR:
(0,0) -> 0.0502
(0,1) -> 0.9999
(1,0) -> 0.9899
(1,1) -> 0.1000
```


In [160]:
import numpy as np

In [161]:
def sigmoid_activation(x):
    """Sigmoid activation function σ(x) = 1 / (1 + e^(-x))."""
    return 1 / (1 + np.exp(-x))

def sigmoid_prime(x):
    """Derivative of the sigmoid function given the *output* of the sigmoid.
    If σ(x) = sigmoid(x), then σ'(x) = σ(x)*(1 - σ(x))."""
    return x * (1 - x)

def tanh_activation(x):
    """Hyperbolic tangent activation function tanh(x)."""
    return np.tanh(x)

def tanh_prime(x):
    """Derivative of the tanh function given the *output* of the tanh.
    If t(x) = tanh(x), then t'(x) = 1 - t(x)^2."""
    return 1 - x ** 2

In [162]:
def get_boolean_data(bool_function):
    """Return input-output pairs for the desired boolean function.
    bool_function can be 'AND', 'OR', 'XOR', or 'ALL'."""
    if bool_function == 'AND':
        # AND truth table
        inputs = np.array([[0, 0],
                           [0, 1],
                           [1, 0],
                           [1, 1]])
        outputs = np.array([[0],
                            [0],
                            [0],
                            [1]])
        return inputs, outputs
    elif bool_function == 'OR':
        # OR truth table
        inputs = np.array([[0, 0],
                           [0, 1],
                           [1, 0],
                           [1, 1]])
        outputs = np.array([[0],
                            [1],
                            [1],
                            [1]])
        return inputs, outputs
    elif bool_function == 'XOR':
        # XOR truth table
        inputs = np.array([[0, 0],
                           [0, 1],
                           [1, 0],
                           [1, 1]])
        outputs = np.array([[0],
                            [1],
                            [1],
                            [0]])
        return inputs, outputs
    elif bool_function == 'ALL':
        # Combine the sets of inputs for AND, OR, XOR (12 total rows)
        inputs = np.array([
            [0, 0, 0, 0],
            [0, 1, 0, 0],
            [1, 0, 0, 0],
            [1, 1, 0, 0],
            
            [0, 0, 0, 1],
            [0, 1, 0, 1],
            [1, 0, 0, 1],
            [1, 1, 0, 1],
            
            [0, 0, 1, 0],
            [0, 1, 1, 0],
            [1, 0, 1, 0],
            [1, 1, 1, 0]
        ])
        outputs = np.array([
            [0], [0], [0], [1],
            [0], [1], [1], [1],
            [0], [1], [1], [0]
        ])
        return inputs, outputs
    else:
        raise ValueError(f"Unknown boolean function: {bool_function}")

In [163]:
def forward_propagate(
    x_values, weight_matrices, bias_vectors, activation_func
):
    """Perform forward propagation through all layers.
    Return the list of layer outputs and layer inputs."""
    layer_inputs = []  # z-values
    layer_outputs = []  # activations

    current_output = x_values

    # For each layer
    for idx in range(len(weight_matrices)):
        layer_input = np.dot(current_output, weight_matrices[idx]) + bias_vectors[idx]
        layer_inputs.append(layer_input)
        current_output = activation_func(layer_input)
        layer_outputs.append(current_output)

    return layer_outputs, layer_inputs

In [164]:
def backward_propagate(
    x_values,
    y_values,
    weight_matrices,
    bias_vectors,
    learning_rate,                
    activation_deriv,
    layer_outputs,
    layer_inputs
):
    """
    Perform backward propagation to update weights and biases.

        W_j <- W_j + learning_rate * Err * g'(in) * x_j
    """

    error = y_values - layer_outputs[-1]
    
    output_delta = error * activation_deriv(layer_outputs[-1])
    
    deltas = [output_delta]

    # Backpropagate delta through hidden layers
    for i in range(len(weight_matrices) - 2, -1, -1):
        # Multiply the current delta by the transpose of the next layer's weights,
        # then multiply by the derivative of this layer's activation
        hidden_delta = deltas[0].dot(weight_matrices[i + 1].T) \
                       * activation_deriv(layer_outputs[i])
        deltas.insert(0, hidden_delta)

    for i in range(len(weight_matrices)):
        if i == 0:
            # The first layer weights get input from x_values
            grad_weights = np.dot(x_values.T, deltas[i])
        else:
            # Subsequent layers get input from the previous layer's output
            grad_weights = np.dot(layer_outputs[i - 1].T, deltas[i])

        # Sum deltas across the batch for bias gradients
        grad_biases = np.sum(deltas[i], axis=0, keepdims=True)

        # -- "Plus" update to match the screenshot's W_j <- W_j + learning_rate * ...
        weight_matrices[i] += learning_rate * grad_weights
        bias_vectors[i]   += learning_rate * grad_biases

    return weight_matrices, bias_vectors

In [165]:
def train_network(
    x_values,
    y_values,
    num_epochs,
    learning_rate,
    weight_matrices,
    bias_vectors,
    activation_func,
    activation_deriv
):
    for _ in range(num_epochs):
        layer_outputs, layer_inputs = forward_propagate(
            x_values, weight_matrices, bias_vectors, activation_func
        )
        weight_matrices, bias_vectors = backward_propagate(
            x_values,
            y_values,
            weight_matrices,
            bias_vectors,
            learning_rate,
            activation_deriv,
            layer_outputs,
            layer_inputs
        )
    return weight_matrices, bias_vectors

In [166]:
def run_network(
    bool_function="ALL",
    activation_choice=0,
    num_hidden_layers=1,       
    num_neurons_per_layer=4,   
    learning_rate=0.1,                    
    num_epochs=10000      
):
    """
    Main function that:
      - Chooses one of (AND, OR, XOR, or ALL).
      - Chooses activation function: 0 -> sigmoid, 1 -> tanh.
      - Creates weights and biases according to the number of hidden layers.
      - Trains the network using backpropagation.
      - Returns the final output (rounded to 4 decimal places).
    """
    np.random.seed(42)

    if activation_choice == 0:
        act_func = sigmoid_activation
        act_deriv = sigmoid_prime
    else:
        act_func = tanh_activation
        act_deriv = tanh_prime

    X, y = get_boolean_data(bool_function)

    weight_matrices = []
    bias_vectors = []

    # If there are no hidden layers, connect the input directly to the output
    if num_hidden_layers == 0:
        weight_matrices.append(
            np.random.uniform(-0.05, 0.05, (X.shape[1], 1))
        )
        bias_vectors.append(np.zeros((1, 1)))
    else:
        # First hidden layer
        weight_matrices.append(
            np.random.uniform(-0.05, 0.05, (X.shape[1], num_neurons_per_layer))
        )
        bias_vectors.append(np.zeros((1, num_neurons_per_layer)))

        for _ in range(num_hidden_layers - 1):
            weight_matrices.append(
                np.random.uniform(-0.05, 0.05, (num_neurons_per_layer, num_neurons_per_layer))
            )
            bias_vectors.append(np.zeros((1, num_neurons_per_layer)))

        # Output layer
        weight_matrices.append(
            np.random.uniform(-0.05, 0.05, (num_neurons_per_layer, 1))
        )
        bias_vectors.append(np.zeros((1, 1)))

    weight_matrices, bias_vectors = train_network(
        X, y,
        num_epochs,
        learning_rate,
        weight_matrices,
        bias_vectors,
        act_func,
        act_deriv
    )

    layer_outputs, _ = forward_propagate(X, weight_matrices, bias_vectors, act_func)
    final_output = layer_outputs[-1]
    return np.round(np.abs(final_output), decimals=4)


In [167]:
bool_function = "ALL"        # AND, OR, XOR, or ALL
activation_choice = 0         # 0 for sigmoid, 1 for tanh
num_hidden_layers = 1
num_neurons_per_layer = 4
learning_rate = 0.1
num_epochs = 10000

outputs = run_network(bool_function, activation_choice, num_hidden_layers, num_neurons_per_layer, learning_rate, num_epochs)

and_part = outputs[0:4]
or_part = outputs[4:8]
xor_part = outputs[8:12]

inputs_2bits = [(0,0), (0,1), (1,0), (1,1)]

print("AND:")
for (inp, outp) in zip(inputs_2bits, and_part):
    print(f"({inp[0]},{inp[1]}) -> {outp[0]:.4f}")

print("\nOR:")
for (inp, outp) in zip(inputs_2bits, or_part):
    print(f"({inp[0]},{inp[1]}) -> {outp[0]:.4f}")

print("\nXOR:")
for (inp, outp) in zip(inputs_2bits, xor_part):
    print(f"({inp[0]},{inp[1]}) -> {outp[0]:.4f}")

AND:
(0,0) -> 0.0007
(0,1) -> 0.0254
(1,0) -> 0.0254
(1,1) -> 0.9557

OR:
(0,0) -> 0.0036
(0,1) -> 0.9843
(1,0) -> 0.9843
(1,1) -> 0.9947

XOR:
(0,0) -> 0.0236
(0,1) -> 0.9556
(1,0) -> 0.9556
(1,1) -> 0.0735


### Допълнителни въпроси
1. **Кога скритите слоеве са строго необходими за да работи невронната мрежа?**
   
   Скритите слоеве са особено необходими, когато входните данни не са линейно разделими. Например, функцията XOR не може да бъде научена само с един изходен слой (т.е. без скрити слоеве), тъй като тя е нелинейно разделима. Скритите слоеве позволяват на невронната мрежа да научава по-сложни зависимости между входовете и изходите.

2. **Какво е значението на "bias" неврона и той влиза ли в съответната бройка неврони? Кои слоеве имат такива неврони?**
   
   "Bias" (байъс) невронът добавя константа към сумата (\(z = w_1 x_1 + \dots + w_n x_n + b\)) преди активационната функция. Тази константа позволява мрежата да се "измества" (shift) и да улавя зависимости, които не минават през началото на координатната система. Обикновено **bias** не се брои в официалния брой неврони, тъй като е просто допълнителен параметър. Всеки слой (освен входния) има байъс неврони, съответно байъс параметри, които се обучават заедно с теглата.
